# Classical ML + 2D ResNet для SER

Признаки: **MFCC** (12 коэф. × 5 статистик), **Pitch/F0** (5 статистик),
**ZCR** (5 статистик), **DWT** (5 уровней × 3 статистики) → **85 признаков**.

Модели: **SVM** (poly), **Random Forest**, **2D ResNet (~10M params)**.  
Вход CNN: MFCC-матрица **(B, 1, 12, 216)** — одноканальное 2D изображение.  
Датасеты: **RESD** (7 классов) и **DUSHA** (5 классов) — обучаются **отдельно**, итого 6 моделей.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'librosa', 'pywavelets', 'datasets', 'soundfile',
    'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'torchinfo',
], check=True)
print('Done.')

## 2. Imports & shared config

In [ ]:
import os, warnings, pathlib
import numpy as np
import pandas as pd
import librosa
import pywt
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED       = 42
SR_TARGET  = 16_000
N_MFCC     = 12
HOP_LENGTH = 512
N_FFT      = 2048
MAX_FRAMES = 216   # для CNN (≈ 6.9 с при hop=512)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Feature extraction

In [ ]:
def _stats(arr):
    return [float(np.min(arr)), float(np.max(arr)),
            float(np.mean(arr)), float(np.median(arr)), float(np.std(arr))]

def extract_mfcc(wav):
    mfcc = librosa.feature.mfcc(y=wav, sr=SR_TARGET, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    return [v for coef in mfcc for v in _stats(coef)]  # 60

def extract_pitch(wav):
    min_lag = int(SR_TARGET / 500)
    max_lag = int(SR_TARGET / 50)
    f0s = []
    for start in range(0, max(1, len(wav) - 2048), HOP_LENGTH):
        frame = wav[start:start + 2048]
        if len(frame) < 2048:
            break
        corr = np.correlate(frame, frame, mode='full')[len(frame) - 1:]
        seg = corr[min_lag:min(max_lag, len(corr))]
        if len(seg) == 0:
            continue
        peak = np.argmax(seg) + min_lag
        f0s.append(SR_TARGET / peak if peak > 0 else 0.0)
    return _stats(np.array(f0s) if f0s else np.array([0.0]))  # 5

def extract_zcr(wav):
    zcr = librosa.feature.zero_crossing_rate(wav, hop_length=HOP_LENGTH)[0]
    return _stats(zcr)  # 5

def extract_dwt(wav):
    coeffs = pywt.wavedec(wav, 'db4', level=4)  # 5 массивов: cA4, cD4..cD1
    feats = []
    for c in coeffs:
        feats += [float(np.std(c)), float(skew(c)), float(kurtosis(c))]
    return feats  # 15

def extract_features(wav):
    return extract_mfcc(wav) + extract_pitch(wav) + extract_zcr(wav) + extract_dwt(wav)  # 85

def extract_mfcc_seq(wav, max_frames=MAX_FRAMES):
    """Для CNN: (max_frames, N_MFCC) с паддингом/обрезкой."""
    mfcc = librosa.feature.mfcc(y=wav, sr=SR_TARGET, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH).T  # (T, 12)
    T = mfcc.shape[0]
    if T >= max_frames:
        return mfcc[:max_frames].astype(np.float32)
    return np.vstack([mfcc, np.zeros((max_frames - T, N_MFCC))]).astype(np.float32)

# имена для RF importance
feature_names = (
    [f'mfcc_{i:02d}_{s}' for i in range(N_MFCC) for s in ['min','max','mean','median','std']]
    + [f'pitch_{s}' for s in ['min','max','mean','median','std']]
    + [f'zcr_{s}'   for s in ['min','max','mean','median','std']]
    + [f'dwt_L{l}_{s}' for l in range(5) for s in ['std','skew','kurt']]
)
print(f'Feature vector: {len(feature_names)} dims')

## 4. Shared training helpers

In [ ]:
from torchinfo import summary as torch_summary

class SeqDataset(Dataset):
    def __init__(self, records):
        self.data = []
        for r in tqdm(records, desc='MFCC seq', leave=False):
            try:
                seq = extract_mfcc_seq(r['wav'])
                self.data.append((seq, r['label']))
            except Exception:
                pass
    def __len__(self):  return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.tensor(x), torch.tensor(y, dtype=torch.long)


class ResBlock2D(nn.Module):
    def __init__(self, channels, kernel=(3, 5), dropout=0.1):
        super().__init__()
        pad = (kernel[0] // 2, kernel[1] // 2)
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.net(x) + x)


class ResTransition2D(nn.Module):
    def __init__(self, in_ch, out_ch, stride, kernel=(3, 5)):
        super().__init__()
        pad = (kernel[0] // 2, kernel[1] // 2)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, (1, 1), stride=stride, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.net(x) + self.skip(x))


class AudioResCNN2D(nn.Module):
    """
    2D ResNet (~10.1M params). Вход: (B, T, n_mfcc) → (B, 1, n_mfcc, T).
      InitConv  : Conv2d(1→64, 3×7)          → (B,  64, 12, 216)
      Stage 1   : 3 × ResBlock2D(64)         → (B,  64, 12, 216)
      Transition: 64→128, stride (2,4)        → (B, 128,  6,  54)
      Stage 2   : 4 × ResBlock2D(128)        → (B, 128,  6,  54)
      Transition: 128→256, stride (2,3)       → (B, 256,  3,  18)
      Stage 3   : 3 × ResBlock2D(256)        → (B, 256,  3,  18)
      GAP + Linear(256, num_classes)
    """
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 64, (3, 7), padding=(1, 3), bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
        )
        self.stage1 = nn.Sequential(*[ResBlock2D(64)  for _ in range(3)])
        self.trans1 = ResTransition2D(64,  128, stride=(2, 4))
        self.stage2 = nn.Sequential(*[ResBlock2D(128) for _ in range(4)])
        self.trans2 = ResTransition2D(128, 256, stride=(2, 3))
        self.stage3 = nn.Sequential(*[ResBlock2D(256) for _ in range(3)])
        self.gap        = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1).unsqueeze(1)
        x = self.init_conv(x)
        x = self.stage1(x); x = self.trans1(x)
        x = self.stage2(x); x = self.trans2(x)
        x = self.stage3(x)
        return self.classifier(self.dropout(self.gap(x).flatten(1)))


def print_cnn_summary(num_classes):
    torch_summary(AudioResCNN2D(num_classes),
                  input_size=(1, MAX_FRAMES, N_MFCC),
                  col_names=['output_size', 'num_params'],
                  col_width=24, row_settings=['var_names'], verbose=1)


class EarlyStopping:
    def __init__(self, patience=10, path='best.pt'):
        self.patience = patience; self.path = path
        self.best = -1.0; self.counter = 0; self.stop = False
    def step(self, metric, model):
        if metric > self.best:
            self.best = metric; self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True


def run_experiment(train_recs, dev_recs, test_recs, label_names, tag, out_dir='/kaggle/working'):
    """
    train_recs : обучение моделей
    dev_recs   : early stopping + LR scheduler (не видит test)
    test_recs  : финальная оценка
    """
    out = pathlib.Path(out_dir)
    num_classes = len(label_names)
    results = {}

    # ── Flat features (SVM / RF) ──────────────────────────────────────────────
    print(f'\n[{tag.upper()}] Extracting features...')
    def _feats(recs):
        Xf, yf = [], []
        for r in tqdm(recs, leave=False):
            try: Xf.append(extract_features(r['wav'])); yf.append(r['label'])
            except Exception: pass
        return np.array(Xf, dtype=np.float32), np.array(yf, dtype=np.int64)

    X_tr, y_tr   = _feats(train_recs)
    X_dev, y_dev = _feats(dev_recs)
    X_te, y_te   = _feats(test_recs)

    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_dev_s = scaler.transform(X_dev)
    X_te_s  = scaler.transform(X_te)

    print(f'  train={len(X_tr)}  dev={len(X_dev)}  test={len(X_te)}')

    # SVM
    print(f'[{tag.upper()}] SVM (poly)...')
    svm = SVC(kernel='poly', degree=3, C=1.0, coef0=1.0,
              class_weight='balanced', random_state=SEED)
    svm.fit(X_tr_s, y_tr)
    results['SVM'] = _metrics(y_te, svm.predict(X_te_s))
    _print_metrics(f'{tag.upper()} SVM (test)', results['SVM'])
    print(classification_report(y_te, svm.predict(X_te_s), target_names=label_names, zero_division=0))

    # Random Forest
    print(f'[{tag.upper()}] Random Forest...')
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)
    results['RF'] = _metrics(y_te, rf.predict(X_te_s))
    _print_metrics(f'{tag.upper()} RF (test)', results['RF'])
    print(classification_report(y_te, rf.predict(X_te_s), target_names=label_names, zero_division=0))
    _plot_importance(rf, feature_names, tag, out)

    # ── 2D ResNet ─────────────────────────────────────────────────────────────
    print(f'\n[{tag.upper()}] 2D ResNet — Architecture:')
    print_cnn_summary(num_classes)

    tr_ds  = SeqDataset(train_recs)
    dev_ds = SeqDataset(dev_recs)
    te_ds  = SeqDataset(test_recs)
    tr_ld  = DataLoader(tr_ds,  batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
    dev_ld = DataLoader(dev_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    te_ld  = DataLoader(te_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    cnn  = AudioResCNN2D(num_classes).to(device)
    crit = nn.CrossEntropyLoss()
    opt  = torch.optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
    es   = EarlyStopping(patience=10, path=str(out / f'best_cnn_{tag}.pt'))

    history = []   # (train_loss, dev_wacc)

    for epoch in range(1, 61):
        # train
        cnn.train()
        loss_sum = 0.0
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(cnn(xb), yb); loss.backward(); opt.step()
            loss_sum += loss.item() * len(yb)
        avg_loss = loss_sum / len(tr_ds)

        # dev eval
        cnn.eval()
        ps, ls = [], []
        with torch.no_grad():
            for xb, yb in dev_ld:
                ps.append(cnn(xb.to(device)).argmax(1).cpu().numpy())
                ls.append(yb.numpy())
        dev_wacc = balanced_accuracy_score(np.concatenate(ls), np.concatenate(ps))

        history.append((avg_loss, dev_wacc))
        sch.step(dev_wacc); es.step(dev_wacc, cnn)
        print(f'  Epoch {epoch:3d}  loss={avg_loss:.4f}  dev_wacc={dev_wacc:.4f}'
              f'  lr={opt.param_groups[0]["lr"]:.2e}' + ('  *' if es.counter == 0 else ''),
              flush=True)
        if es.stop:
            print(f'  Early stop at epoch {epoch}'); break

    # final eval on test
    cnn.load_state_dict(torch.load(str(out / f'best_cnn_{tag}.pt'), map_location=device))
    cnn.eval()
    ps, ls = [], []
    with torch.no_grad():
        for xb, yb in te_ld:
            ps.append(cnn(xb.to(device)).argmax(1).cpu().numpy())
            ls.append(yb.numpy())
    cnn_pred, cnn_labels = np.concatenate(ps), np.concatenate(ls)
    results['ResNet'] = _metrics(cnn_labels, cnn_pred)
    _print_metrics(f'{tag.upper()} ResNet (test)', results['ResNet'])
    print(classification_report(cnn_labels, cnn_pred, target_names=label_names, zero_division=0))

    _plot_learning(history, tag, out)
    _plot_cm(cnn_labels, cnn_pred, label_names, tag, out)
    _plot_summary(results, tag, out)
    return results


def _metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'wacc':     balanced_accuracy_score(y_true, y_pred),
            'f1_macro': f1_score(y_true, y_pred, average='macro',    zero_division=0),
            'f1_w':     f1_score(y_true, y_pred, average='weighted', zero_division=0)}

def _print_metrics(title, m):
    print(f'\n=== {title} ===')
    for k, v in m.items(): print(f'  {k:12s}: {v:.4f}')

def _plot_importance(rf, feat_names, tag, out):
    idx = np.argsort(rf.feature_importances_)[::-1][:30]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([feat_names[i] for i in idx[::-1]], rf.feature_importances_[idx[::-1]], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'[{tag.upper()}] RF — Top 30 Feature Importances')
    plt.tight_layout(); plt.savefig(out / f'rf_importance_{tag}.png', dpi=150); plt.show()

def _plot_learning(history, tag, out):
    losses = [h[0] for h in history]
    waccs  = [h[1] for h in history]
    epochs = range(1, len(history) + 1)
    best_epoch = int(np.argmax(waccs)) + 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(epochs, losses, color='steelblue', label='train loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'[{tag.upper()}] Train Loss'); ax1.legend()

    ax2.plot(epochs, waccs, color='darkorange', label='dev wacc')
    ax2.axvline(best_epoch, color='red', linestyle='--', alpha=0.7,
                label=f'best epoch {best_epoch} ({max(waccs):.4f})')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Weighted Accuracy')
    ax2.set_title(f'[{tag.upper()}] Dev Weighted Accuracy'); ax2.legend()

    plt.tight_layout()
    plt.savefig(out / f'cnn_curves_{tag}.png', dpi=150); plt.show()

def _plot_cm(labels, preds, label_names, tag, out):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[0])
    axes[0].set_title(f'[{tag.upper()}] ResNet CM (counts)'); axes[0].set_xlabel('Predicted')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[1])
    axes[1].set_title(f'[{tag.upper()}] ResNet CM (row-norm)'); axes[1].set_xlabel('Predicted')
    plt.tight_layout(); plt.savefig(out / f'cnn_cm_{tag}.png', dpi=150); plt.show()

def _plot_summary(results, tag, out):
    df = pd.DataFrame(results, index=['accuracy','wacc','f1_macro','f1_w']).T
    print(f'\n── [{tag.upper()}] Summary (test) ──')
    print(df.to_string(float_format='{:.4f}'.format))
    df.to_csv(out / f'results_{tag}.csv')

print('Helpers ready.')

## 5. RESD — Load & Run (7 классов)

Сплит: HuggingFace `train` → 80% train / 20% dev.  
Финальная оценка: HuggingFace `test` (280 сэмплов, оригинальный сплит).

In [ ]:
from datasets import load_dataset
from collections import Counter

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_LABELS = ['happiness','sadness','anger','fear','disgust','enthusiasm','neutral']

def _load_resd_split(hf_split):
    recs = []
    for ex in tqdm(hf_split, leave=False):
        audio = ex['speech']
        wav   = np.array(audio['array'], dtype=np.float32)
        sr    = audio['sampling_rate']
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        recs.append({'wav': wav, 'label': RESD_LABEL2ID[ex['emotion']]})
    return recs

ds = load_dataset('Aniemore/resd')
resd_hf_train = _load_resd_split(ds['train'])   # 1116 записей
resd_test      = _load_resd_split(ds['test'])    # 280 записей (финальный тест)

# Делим HF train → 80% train / 20% dev (стратифицированно)
resd_tr_idx, resd_dev_idx = train_test_split(
    range(len(resd_hf_train)), test_size=0.2, random_state=SEED,
    stratify=[r['label'] for r in resd_hf_train])
resd_train = [resd_hf_train[i] for i in resd_tr_idx]
resd_dev   = [resd_hf_train[i] for i in resd_dev_idx]

print(f'RESD  train={len(resd_train)}  dev={len(resd_dev)}  test={len(resd_test)}')
for lid, name in enumerate(RESD_LABELS):
    tr = sum(1 for r in resd_train if r['label'] == lid)
    dv = sum(1 for r in resd_dev   if r['label'] == lid)
    te = sum(1 for r in resd_test  if r['label'] == lid)
    print(f'  {name:12s}  tr={tr:3d}  dev={dv:3d}  test={te:3d}')

In [ ]:
resd_results = run_experiment(resd_train, resd_dev, resd_test, RESD_LABELS, tag='resd')

## 6. DUSHA — Load & Run (5 классов)

Сплит: `aggregated_majority.tsv` (train) → 80% train / 20% dev.  
Финальная оценка: `aggregated_ds_0.9_test.tsv` (отдельный тестовый TSV).

In [ ]:
DUSHA_TRAIN_TSV  = '/kaggle/input/datasets/aleksandribryanov/agg-dusha/aggregated_majority.tsv'
DUSHA_TEST_TSV   = '/kaggle/input/datasets/aleksandribryanov/agg-dusha/aggregated_ds_0.9_test.tsv'
DUSHA_AUDIO_TRAIN = '/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_train'
DUSHA_AUDIO_TEST  = '/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_test'
DUSHA_FRACTION   = 0.1   # 10% от train (~10k записей)

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']

def _load_dusha_tsv(tsv_path, audio_dir, fraction=None):
    df = pd.read_csv(tsv_path, sep='\t')
    df = df[df['aggregated_emo'].isin(DUSHA_LABEL2ID)]
    if fraction is not None:
        df = df.sample(frac=fraction, random_state=SEED)
    recs = []
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        path = pathlib.Path(audio_dir) / row['audio_path']
        if not path.exists():
            continue
        try:
            wav, sr = sf.read(str(path), dtype='float32')
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != SR_TARGET:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
            recs.append({'wav': wav, 'label': DUSHA_LABEL2ID[row['aggregated_emo']]})
        except Exception:
            pass
    return recs

# Загружаем train TSV и делим 80/20
dusha_all  = _load_dusha_tsv(DUSHA_TRAIN_TSV, DUSHA_AUDIO_TRAIN, fraction=DUSHA_FRACTION)
dusha_test = _load_dusha_tsv(DUSHA_TEST_TSV,  DUSHA_AUDIO_TEST,  fraction=None)

dusha_tr_idx, dusha_dev_idx = train_test_split(
    range(len(dusha_all)), test_size=0.2, random_state=SEED,
    stratify=[r['label'] for r in dusha_all])
dusha_train = [dusha_all[i] for i in dusha_tr_idx]
dusha_dev   = [dusha_all[i] for i in dusha_dev_idx]

print(f'DUSHA  train={len(dusha_train)}  dev={len(dusha_dev)}  test={len(dusha_test)}')
for lid, name in enumerate(DUSHA_LABELS):
    tr = sum(1 for r in dusha_train if r['label'] == lid)
    dv = sum(1 for r in dusha_dev   if r['label'] == lid)
    te = sum(1 for r in dusha_test  if r['label'] == lid)
    print(f'  {name:12s}  tr={tr:4d}  dev={dv:4d}  test={te:4d}')

In [ ]:
dusha_results = run_experiment(dusha_train, dusha_dev, dusha_test, DUSHA_LABELS, tag='dusha')

## 7. Итоговое сравнение всех 6 моделей

In [ ]:
rows = []
for model_name in ['SVM', 'RF', 'ResNet']:
    for dataset, res in [('RESD', resd_results), ('DUSHA', dusha_results)]:
        m = res[model_name]
        rows.append({
            'Dataset': dataset, 'Model': model_name,
            'Accuracy':    round(m['accuracy'], 4),
            'WAcc':        round(m['wacc'],     4),
            'F1 Macro':    round(m['f1_macro'], 4),
            'F1 Weighted': round(m['f1_w'],     4),
        })

df_all = pd.DataFrame(rows).set_index(['Dataset', 'Model'])
print(df_all.to_string())
df_all.to_csv('/kaggle/working/results_all.csv')

fig, ax = plt.subplots(figsize=(9, 4))
x      = np.arange(3)
width  = 0.35
models = ['SVM', 'RF', 'ResNet']
resd_w  = [resd_results[m]['wacc']  for m in models]
dusha_w = [dusha_results[m]['wacc'] for m in models]
ax.bar(x - width/2, resd_w,  width, label='RESD',  color='steelblue')
ax.bar(x + width/2, dusha_w, width, label='DUSHA', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylabel('Weighted Accuracy')
ax.set_title('Weighted Accuracy — все 6 моделей')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/kaggle/working/results_comparison.png', dpi=150)
plt.show()